## Load silver table data

In [0]:
%pip install sentence-transformers scikit-learn
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.table("virtue_foundation_dataset.silver.facilities_cleaned")

# Only keep valid rows
df = df.filter(col("audit_reason").isNull())

## GOLD TABLES

In [0]:
facilities_core = df.select(
    "unique_id",
    "name",
    "organization_type",
    "facilityTypeId",
    "operatorTypeId",
    "yearEstablished",
    "capacity",
    "numberDoctors"
)

facilities_core.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities_core")

facilities_location = df.select(
    "unique_id",
    "address_line1",
    "address_line2",
    "address_city",
    "address_stateOrRegion",
    "address_country",
    "latitude",
    "longitude"
)

facilities_location.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities")

contact = df \
    .withColumn("phone", explode_outer("phone_numbers")) \
    .withColumn("website", explode_outer("websites")) \
    .select("unique_id", "phone", "website", "email")

contact.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities_contact")


## One hot encoding

In [0]:
array_cols = [
    "specialties", "procedure", "equipment", "capability"
]

# Optional: only handle nulls safely (no type changes)
for c in array_cols:
    df = df.withColumn(
        c,
        when(col(c).isNull(), array()).otherwise(col(c))
    )


from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

from sklearn.cluster import KMeans
import pandas as pd
import numpy as np

def ai_cluster_ohe(df, colname, table_prefix, k=20):
    
    # 1. explode
    exploded = df.select(
        "unique_id",
        explode_outer(col(colname)).alias("raw")
    ).dropna()

    # 2. move to pandas (small-ish distinct values)
    distinct_vals = exploded.select("raw").distinct().toPandas()

    values = distinct_vals["raw"].tolist()

    # 3. embed
    embeddings = model.encode(values)

    # 4. cluster
    kmeans = KMeans(n_clusters=min(k, len(values)), random_state=42)
    clusters = kmeans.fit_predict(embeddings)

    distinct_vals["cluster"] = clusters

    # 5. map back
    mapping = spark.createDataFrame(distinct_vals)

    clustered = exploded.join(mapping, "raw", "left")

    # save semantic clustered table
    clustered.write.mode("overwrite").saveAsTable(f"gold.{table_prefix}")

    # 6. OHE
    ohe = clustered.groupBy("unique_id") \
        .pivot("cluster") \
        .agg(lit(1)) \
        .na.fill(0)

    # rename cols
    for c in ohe.columns:
        if c != "unique_id":
            ohe = ohe.withColumnRenamed(c, f"{table_prefix.upper()}_{c}")

    ohe.write.mode("overwrite").saveAsTable(f"gold.{table_prefix}_ohe")

    return clustered, ohe


## OHE specific tables

In [0]:
equipment_df, equipment_ohe = ai_cluster_ohe(
    df, "equipment", "facilities_equipment", k=30
)

specialties_df, specialties_ohe = ai_cluster_ohe(
    df, "specialties", "facilities_specialties", k=30
)

procedures_df, procedures_ohe = ai_cluster_ohe(
    df, "procedure", "facilities_procedures", k=40
)

capabilities_df, capabilities_ohe = ai_cluster_ohe(
    df, "capability", "facilities_capabilities", k=30
)

## Core Table and Final Feature Table

In [0]:
facilities_core = df.select(
    "unique_id", "name", "organization_type",
    "facilityTypeId", "capacity", "numberDoctors"
)

facilities_core.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities_core")

gold_features = facilities_core \
    .join(equipment_ohe, "unique_id", "left") \
    .join(specialties_ohe, "unique_id", "left") \
    .join(procedures_ohe, "unique_id", "left") \
    .join(capabilities_ohe, "unique_id", "left") \
    .na.fill(0)

gold_features.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities_features")